# MagicBox: full LFM2.5 Encoder training

Run on Kaggle TPU v5e-8 or a Colab runtime exposing 8 TPU devices.
Attach the completed MagicBox dataset from the CPU data notebook and enable internet.
This notebook contains its training code. It installs Python 3.12 and pinned dependencies,
downloads the encoder, checks a tiny model, then trains every parameter for 3 epochs.

Outputs go to `magicbox-run`: resumable optimizer checkpoints, validation reports,
and a complete inference bundle. Each session runs for up to 8 hours.
To continue in a new session, attach the previous output and set `RESUME_FROM` below.
Checkpoints retain the 2 newest complete states. Training continues at the next unread batch.


In [ ]:
from pathlib import Path
import os
import json
import subprocess
import sys

IS_KAGGLE = Path('/kaggle/working').is_dir()
HOST = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
SCRATCH = Path('/kaggle/temp/magicbox') if IS_KAGGLE else HOST / 'magicbox-scratch'
OUTPUT = HOST / 'magicbox-run'
DATASET = None  # Set Path('/kaggle/input/your-data/magicbox') if multiple datasets are attached.
RESUME_FROM = None  # Path to a previous magicbox-run folder.
EPOCHS = 3
SESSION_HOURS = 8
REQUESTS = 8
MICROBATCHES = 4  # Global logical batch: 32 requests.
ROW_CHUNK = 4
LEARNING_RATE = 2e-5
VALIDATION_RECORDS = 256
FINAL_RECORDS = 0  # 0 evaluates every held-out record when all epochs finish.
SCRATCH.mkdir(parents=True, exist_ok=True)
CHECKOUT = SCRATCH / 'training'
VENV = SCRATCH / 'venv'
PYTHON = VENV / 'bin/python'
MODEL = SCRATCH / 'encoder'
CACHE = SCRATCH / 'arrow'

def run_child(*command, env=None):
    with subprocess.Popen(command, cwd=CHECKOUT, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)


In [ ]:
import base64
import hashlib
import io
import zipfile

SOURCE_SHA256 = 'bd5c48c73b02d37b5fd263e1f796db5da4023e9a055a039506f32ff934a913dc'
SOURCE_ARCHIVE = '...'
archive_bytes = base64.b64decode(SOURCE_ARCHIVE)
assert hashlib.sha256(archive_bytes).hexdigest() == SOURCE_SHA256
CHECKOUT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        target = (CHECKOUT / member.filename).resolve()
        assert target.is_relative_to(CHECKOUT.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
print('Training source snapshot:', SOURCE_SHA256)


In [ ]:
UV = SCRATCH / 'tooling/bin/uv'
if not UV.is_file():
    run_child(sys.executable, '-m', 'pip', 'install', '--target',
              str(SCRATCH / 'tooling'), 'uv==0.11.30')
environment = {**os.environ, 'UV_PROJECT_ENVIRONMENT': str(VENV),
               'UV_CACHE_DIR': str(SCRATCH / 'uv-cache')}
run_child(str(UV), 'sync', '--locked', '--no-dev', '--all-extras',
          '--python', '3.12', env=environment)
run_child(str(UV), 'pip', 'install', '--python', str(PYTHON), 'jax[tpu]==0.7.2')
run_child(str(PYTHON), '-c', "import jax; print(jax.__version__, jax.devices()); assert jax.__version__ == '0.7.2'; assert jax.process_count() == 1; assert jax.device_count() == jax.local_device_count() == 8; assert all(d.platform == 'tpu' for d in jax.devices())")


## Dataset and encoder

The loader checks every shard checksum and the exact tokenizer contract. It preserves
train, validation, calibration, test, and OOD splits. Source and schema lengths are
bucketed without truncation; candidate counts use operational padding limits.


In [ ]:
import shutil
if DATASET is None:
    search = Path('/kaggle/input') if IS_KAGGLE else HOST
    candidates = []
    for path in search.rglob('manifest.json'):
        try:
            manifest = json.loads(path.read_text())
            if manifest.get('format') == 'minifield.magicbox/1.0' and manifest.get('complete') is True:
                candidates.append(path.parent)
        except (OSError, ValueError):
            pass
    assert len(candidates) == 1, f'Set DATASET to one completed dataset directory: {candidates}'
    DATASET = candidates[0]
DATASET = Path(DATASET)
if RESUME_FROM is not None and not OUTPUT.exists():
    shutil.copytree(RESUME_FROM, OUTPUT)
manifest = json.loads((DATASET / 'manifest.json').read_text())
assert manifest['mode'] == 'full'
SOURCE_TOKENS = manifest['tokenizer']['source_limit']
SCHEMA_TOKENS = manifest['tokenizer']['schema_limit']
by_split = {}
for shard in manifest['shards']:
    by_split[shard['split']] = by_split.get(shard['split'], 0) + shard['rows']
print('Dataset:', DATASET, '\nRecords:', by_split)
print('Logical updates per epoch:', (by_split['train'] + REQUESTS * MICROBATCHES - 1) // (REQUESTS * MICROBATCHES))
download = "from huggingface_hub import snapshot_download; from minifield_training.models.lfm2_5 import encoder; snapshot_download(encoder.SOURCE.model_id, revision=encoder.SOURCE.revision, allow_patterns=['config.json','tokenizer.json','model.safetensors'], local_dir=" + repr(str(MODEL)) + ")"
run_child(str(PYTHON), '-c', download)


## Startup checks

First, a tiny CPU model learns handwritten examples for all 4 types and verifies exact
checkpoint recovery. Then the full encoder commits 2 TPU updates, saves a checkpoint,
and evaluates 8 validation records. Those updates become the start of the full run.


In [ ]:
run_child(str(PYTHON), '-m', 'examples.magicbox.smoke',
          env={**os.environ, 'JAX_PLATFORMS': 'cpu'})
command = [str(PYTHON), '-m', 'examples.magicbox.train',
           '--dataset', str(DATASET), '--model-dir', str(MODEL),
           '--output', str(OUTPUT), '--cache', str(CACHE),
           '--devices', '8', '--platform', 'tpu', '--requests', str(REQUESTS),
           '--microbatches', str(MICROBATCHES), '--row-chunk', str(ROW_CHUNK),
           '--source-tokens', str(SOURCE_TOKENS), '--schema-tokens', str(SCHEMA_TOKENS),
           '--epochs', str(EPOCHS), '--learning-rate', str(LEARNING_RATE)]
if not list((OUTPUT / 'checkpoints').glob('step-*/manifest.json')):
    run_child(*command, '--max-steps', '2', '--max-hours', '1',
              '--checkpoint-every', '2', '--validation-records', '8')


## Full training

BF16 activations, FP32 master weights and Adam moments, gradient checkpointing,
and 8-device data parallelism. Losses average fields within each active task type,
then give each type equal weight. AdamW uses gradient clipping at 1.0 and weight decay 0.01.
Validation runs every 250 committed updates. Final evaluation covers all available holdouts.


In [ ]:
run_child(*command, '--max-hours', str(SESSION_HOURS), '--checkpoint-every', '250',
          '--validation-records', str(VALIDATION_RECORDS), '--final-records', str(FINAL_RECORDS))


## Reload and predict

This loads the exported bundle in a fresh process and runs the complete typed decoder.
Change the request to inspect extraction, choice, boolean probabilities, and ordinal scores.


In [ ]:
request = {'state': 'Ada sent 3 reports.', 'questions': {
    'person': {'type': 'extract', 'instructions': 'Who sent the reports?'},
    'topic': {'type': 'choice', 'instructions': 'Classify the message.',
              'criteria': {'reporting': 'Reports and documents', 'payments': 'Billing and payments'}},
    'sent': {'type': 'noul', 'instructions': 'Were reports sent?'},
    'volume': {'type': 'score', 'instructions': 'How many reports were sent?',
               'criteria': ['None', 'One', 'Several']}}}
request_path = OUTPUT / 'sample-request.json'
request_path.write_text(json.dumps(request))
bundle = sorted(OUTPUT.glob('bundle-*'))[-1]
run_child(str(PYTHON), '-m', 'examples.magicbox.predict', '--bundle', str(bundle),
          '--request', str(request_path))
print('Keep this folder for inference:', bundle)
print('Keep this folder to resume training:', OUTPUT / 'checkpoints')
for path in sorted(OUTPUT.glob('final-*.json')):
    print(path.name, json.loads(path.read_text()))
